# Figure 2 — variant length spectrum (merge)

Merge per-chromosome outputs from `VariantLengthSpectrum.wdl` launched on
the Terra `GL_INTERVAL_set` table (DeepVariant + GLnexus, as in Table 2).
`MergeVariantLengthSpectrum.wdl` already scans the v3 SV companions and merges;
to merge here instead, put its `companions.length_spectrum.{bins.tsv,summary.json}`
in `summaries/fig2/shards/` too.

1. Stage scripts: `gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/"`
2. Run the WDL (see `variant_length/README.md`)
3. Set `FIG2_RUN_PIPELINE=true` here to pull shard outputs and merge

## Outputs

- `summaries/fig2/fig2_length_spectrum.bins.tsv`
- `summaries/fig2/fig2_length_spectrum.summary.json` (check `table2_check`)

Copy the merged TSV into the manuscript repo:
`aou-lr-phase-2-manuscript/mockup_figures/data/fig2_length_spectrum.bins.tsv`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "merge_variant_length_spectrum.py").is_file() or (_d / "terra_notebook.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
if _scripts and str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))
print("scripts:", _scripts)


In [ ]:
import json
from pathlib import Path

RUN = os.environ.get("FIG2_RUN_PIPELINE", "false").strip().lower() in {"1", "true", "yes", "on"}
OUT_DIR = Path(os.environ.get("FIG2_OUT_DIR", "summaries/fig2")).resolve()
SHARD_DIR = OUT_DIR / "shards"
TABLE = os.environ.get("FIG2_TABLE", "GL_INTERVAL_set")
NAMESPACE = os.environ.get("WORKSPACE_NAMESPACE", "allofus-drc-wgs-LR-prodData")
WORKSPACE = os.environ.get("WORKSPACE_NAME", "AoU_DRC_LongReads_PhaseTwo_Storage")

OUT_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print({"RUN": RUN, "OUT_DIR": str(OUT_DIR), "TABLE": TABLE})


In [ ]:
# Pull shard outputs from the data table (bins + summary columns written by the WDL).
if RUN:
    from terra_notebook import load_entity_table  # type: ignore

    df = load_entity_table(NAMESPACE, WORKSPACE, TABLE)
    bin_col = next((c for c in ("bins", "length_spectrum_bins") if c in df.columns), None)
    sum_col = next((c for c in ("summary", "length_spectrum_summary") if c in df.columns), None)
    if bin_col is None or sum_col is None:
        raise RuntimeError(
            f"Table {TABLE} missing bins/summary columns; have {list(df.columns)}. "
            "Map WDL outputs onto the data table, or place shard files under summaries/fig2/shards/."
        )
    for _, row in df.iterrows():
        chrom = row.get(f"entity:{TABLE}_id") or row.get(f"{TABLE}_id") or row.name
        for col, suffix in ((bin_col, "bins.tsv"), (sum_col, "summary.json")):
            uri = row[col]
            if not isinstance(uri, str) or not uri.startswith("gs://"):
                continue
            dest = SHARD_DIR / f"{chrom}.length_spectrum.{suffix}"
            if dest.exists() and os.environ.get("FIG2_FORCE", "").lower() not in {"1", "true"}:
                continue
            print("gsutil cp", uri, dest)
            subprocess.check_call(["gsutil", "-q", "cp", uri, str(dest)])
else:
    print("Set FIG2_RUN_PIPELINE=true to download shards from Terra")


In [ ]:
bins = sorted(SHARD_DIR.glob("*.length_spectrum.bins.tsv"))
summaries = sorted(SHARD_DIR.glob("*.length_spectrum.summary.json"))
print(f"{len(bins)} bin shards, {len(summaries)} summaries")
if bins and summaries:
    out_bins = OUT_DIR / "fig2_length_spectrum.bins.tsv"
    out_sum = OUT_DIR / "fig2_length_spectrum.summary.json"
    cmd = [
        sys.executable,
        str(_scripts / "merge_variant_length_spectrum.py"),
        "--bins",
        *[str(p) for p in bins],
        "--summaries",
        *[str(p) for p in summaries],
        "--out-bins",
        str(out_bins),
        "--out-summary",
        str(out_sum),
    ]
    subprocess.check_call(cmd)
    print(out_bins.read_text(encoding="utf-8").splitlines()[:8])
    print(json.loads(out_sum.read_text(encoding="utf-8")))
else:
    print("No shards yet — run the WDL first")
